In [19]:
# =============================================================================
# Exercise prob020903 – Solution of the State Equation
# Pure time-domain approach: Computation of exp(At) using two methods
# No Laplace transform or s-domain used
# Suitable for Springer-style supplementary material
# =============================================================================

import numpy as np

# ───────────────────────────────────────────────
# Given system parameters
# ───────────────────────────────────────────────

A = np.array([[ 0,  2], [-1, -3]], dtype=float)
B = np.array([[0],[1]], dtype=float)
q0 = np.array([[2], [1]], dtype=float) 

print("System matrix A:")
print(A)
print("\nInput vector B:")
print(B)
print("\nInitial state q(0):")
print(q0)

print("\n" + "="*70)
print("The state equation is:  q̇(t) = A q(t) + B x(t)")
print("With x(t) = 0, the solution simplifies to:")
print("q(t) = exp(A t) q(0)\n")

# ───────────────────────────────────────────────
# Characteristic polynomial & eigenvalues (computed dynamically)
# ───────────────────────────────────────────────

char_poly_coeffs = np.poly(A)  # highest degree first
print("Characteristic polynomial coefficients (highest degree first):")
print("   ", char_poly_coeffs)

# Pretty-print polynomial
poly_terms = []
for i, coeff in enumerate(char_poly_coeffs):
    deg = len(char_poly_coeffs) - 1 - i
    if coeff == 0:
        continue
    sign = " + " if coeff > 0 and i > 0 else " - " if coeff < 0 and i > 0 else ""
    abs_coeff = abs(coeff)
    coeff_str = f"{abs_coeff:g}" if abs_coeff != 1 or deg == 0 else ""
    if deg == 0:
        poly_terms.append(f"{sign}{coeff_str}")
    elif deg == 1:
        poly_terms.append(f"{sign}{coeff_str}λ")
    else:
        poly_terms.append(f"{sign}{coeff_str}λ^{deg}")

poly_str = "".join(poly_terms).lstrip(" +").lstrip(" -")
print(f"Characteristic equation: det(A - λI) = {poly_str} = 0")

eigenvalues = np.roots(char_poly_coeffs)
eigenvalues = np.sort(eigenvalues)[::-1]  # descending order

print("\nEigenvalues:")
for i, lam in enumerate(eigenvalues, 1):
    print(f"  λ_{i} = {lam:.4g}")

print()

# ───────────────────────────────────────────────
# Method 1: β coefficients (Cayley–Hamilton approach)
# ───────────────────────────────────────────────

print("Method 1 – Cayley–Hamilton / β-coefficients approach\n")

V = np.array([[1, eigenvalues[0]], [1, eigenvalues[1]]], dtype=float)
V_inv = np.linalg.inv(V)

print("Vandermonde matrix V:")
print(V)
print("\nInverse V⁻¹:")
print(V_inv)

lam1 = eigenvalues[0]   # -1.0
lam2 = eigenvalues[1]   # -2.0

print("\nTherefore (with actual eigenvalues λ₁ = {:.0f}, λ₂ = {:.0f}):".format(lam1, lam2))
print(f"  β₀(t) = {V_inv[0,0]:+.4g} e^{{{lam1:.0f}t}}  {V_inv[0,1]:+.4g} e^{{{lam2:.0f}t}}")
print(f"  β₁(t) = {V_inv[1,0]:+.4g} e^{{{lam1:.0f}t}}  {V_inv[1,1]:+.4g} e^{{{lam2:.0f}t}}")

def beta(t):
    exp_lam_t = np.exp(eigenvalues * t)
    return V_inv @ exp_lam_t

# ───────────────────────────────────────────────
# Method 2: Diagonalization (using exact eigenvectors from the text)
# ───────────────────────────────────────────────

print("\n" + "="*70)
print("Method 2 – Diagonalization / modal decomposition\n")

P = np.array([[-2,  1], [ 1, -1]], dtype=float)
P_inv = np.linalg.inv(P)

print("Modal matrix P (using eigenvectors from the solution):")
print(P)
print("\nMatrix P⁻¹:")
print(P_inv)

def exp_At(t, method=1):
    if method == 1:
        b = beta(t)
        return b[0] * np.eye(2) + b[1] * A
    else:
        D_t = np.diag(np.exp(eigenvalues * t))
        return P @ D_t @ P_inv

t_test = 1.0
print(f"\nexp(At) at t = {t_test} (Method 1):")
print(np.round(exp_At(t_test, 1), 6))
print(f"exp(At) at t = {t_test} (Method 2):")
print(np.round(exp_At(t_test, 2), 6))

print("\nBoth methods agree within floating-point precision.\n")

# ───────────────────────────────────────────────
# Final state vector q(t) – fully computed coefficients
# ───────────────────────────────────────────────

print("\n" + "="*70)
print("Final result – State vector q(t)\n")

v0 = q0           # shape (2,1) – coefficient of β₀(t)
v1 = A @ q0       # shape (2,1) – coefficient of β₁(t)

print("q(t) = β₀(t) ⋅ q(0) + β₁(t) ⋅ (A q(0))")
print(f"  q(0)     =\n{v0}")
print(f"  A q(0)   =\n{v1}\n")

# Coefficients for e^{λ₁ t} and e^{λ₂ t}
c01, c02 = V_inv[0]     # coeffs for β₀(t)
c11, c12 = V_inv[1]     # coeffs for β₁(t)

q1_exp1 = v0[0,0] * c01 + v1[0,0] * c11
q1_exp2 = v0[0,0] * c02 + v1[0,0] * c12

q2_exp1 = v0[1,0] * c01 + v1[1,0] * c11
q2_exp2 = v0[1,0] * c02 + v1[1,0] * c12

print("Resulting expressions (fully computed):")
print(f"  q₁(t) = {q1_exp1:+.0f} e^{{λ₁ t}}  {q1_exp2:+.0f} e^{{λ₂ t}}")
print(f"  q₂(t) = {q2_exp1:+.0f} e^{{λ₁ t}}  {q2_exp2:+.0f} e^{{λ₂ t}}")

print("\nSubstituting λ₁ = -1, λ₂ = -2:")
print(f"  q₁(t) = {q1_exp1:+.0f} e^{{-t}}  {q1_exp2:+.0f} e^{{-2t}}")
print(f"  q₂(t) = {q2_exp1:+.0f} e^{{-t}}  {q2_exp2:+.0f} e^{{-2t}}")

print("\nEnd of solution.")

System matrix A:
[[ 0.  2.]
 [-1. -3.]]

Input vector B:
[[0.]
 [1.]]

Initial state q(0):
[[2.]
 [1.]]

The state equation is:  q̇(t) = A q(t) + B x(t)
With x(t) = 0, the solution simplifies to:
q(t) = exp(A t) q(0)

Characteristic polynomial coefficients (highest degree first):
    [1. 3. 2.]
Characteristic equation: det(A - λI) = λ^2 + 3λ + 2 = 0

Eigenvalues:
  λ_1 = -1
  λ_2 = -2

Method 1 – Cayley–Hamilton / β-coefficients approach

Vandermonde matrix V:
[[ 1. -1.]
 [ 1. -2.]]

Inverse V⁻¹:
[[ 2. -1.]
 [ 1. -1.]]

Therefore (with actual eigenvalues λ₁ = -1, λ₂ = -2):
  β₀(t) = +2 e^{-1t}  -1 e^{-2t}
  β₁(t) = +1 e^{-1t}  -1 e^{-2t}

Method 2 – Diagonalization / modal decomposition

Modal matrix P (using eigenvectors from the solution):
[[-2.  1.]
 [ 1. -1.]]

Matrix P⁻¹:
[[-1. -1.]
 [-1. -2.]]

exp(At) at t = 1.0 (Method 1):
[[ 0.600424  0.465088]
 [-0.232544 -0.097209]]
exp(At) at t = 1.0 (Method 2):
[[ 0.600424  0.465088]
 [-0.232544 -0.097209]]

Both methods agree within float